In [ ]:
import pandas as pd
import os
import glob

conditions = {
    'wt': 'wt',
    'mutant': 'fj28/selected random files'
}

def load_condition(directory, condition_name, particle_offset_base):
    csv_files = sorted(glob.glob(os.path.join(directory, '*.csv')))
    
    if not csv_files:
        print(f"WARNING: No CSV files found in '{directory}'")
        return pd.DataFrame()
    
    dfs = []
    for file_idx, filepath in enumerate(csv_files):
        df = pd.read_csv(filepath)
        filename = os.path.basename(filepath)
        
        df['source_file'] = filename
        df['file_idx']    = file_idx
        df['condition']   = condition_name
        
        # Drop the column frame.1 if it's a duplicate of frame (sometimes comes if rabitpy og code used)
        if 'frame.1' in df.columns:
            if (df['frame'] == df['frame.1']).all():
                df = df.drop(columns=['frame.1'])
            else:
                print(f"  WARNING: frame.1 != frame in {filename} — keeping both for inspection")
        
        # Unique particle IDs across files AND conditions
        max_particle_offset = 10_000
        df['particle'] = df['particle'] + particle_offset_base + file_idx * max_particle_offset
        
        dfs.append(df)
        print(f"  Loaded: {filename} — {len(df)} rows, {df['particle'].nunique()} particles")
    
    return pd.concat(dfs, ignore_index=True)


all_dfs = []

print("Loading WT...")
wt_df = load_condition(conditions['wt'], 'wt', particle_offset_base=0)
all_dfs.append(wt_df)

print("\nLoading Mutant...")
# Offset mutant particles index far from wt to avoid any overlap errors
# maybe wt gets 0–999,999 and mutant starts at 1,000,000 ??
mutant_df = load_condition(conditions['mutant'], 'mutant', particle_offset_base=1_000_000)
all_dfs.append(mutant_df)

combined = pd.concat(all_dfs, ignore_index=True)

# Summary
print("\n--- Summary ---")
print(f"Total rows:       {len(combined)}")
print(f"Unique particles: {combined['particle'].nunique()}")
print(f"\nPer condition:")
print(combined.groupby('condition')['particle'].nunique().rename('unique_particles').to_frame())
print(f"\nColumns: {list(combined.columns)}")
print(combined.head())

combined.to_csv('wt_fj28_final_tracks_for_scm.csv', index=False)
print("\nSaved to all_conditions_combined.csv")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors
import numpy as np

# define colormap
from matplotlib.colors import LinearSegmentedColormap

def _dark_saturated_cmap():
    """
    Custom colormap: dark & saturated throughout, no pastels/light regions.
    dark red → burnt orange → dark gold → forest green → teal → navy → indigo/purple
    """
    colors = [
        '#8B0000',  # dark red
        '#CC3300',  # deep vermillion  
        '#CC6600',  # burnt orange
        '#B8860B',  # dark goldenrod
        '#2E8B00',  # forest green
        '#007060',  # dark teal
        '#005599',  # strong blue
        '#1A1A8B',  # navy
        '#4B0082',  # indigo
    ]
    return LinearSegmentedColormap.from_list('dark_saturated', colors, N=256)




def plot_trajectories(data, condition_col='condition', particle_col='particle',
                      x_col='x', y_col='y', pixel_to_micron=0.1666666,
                      xlim=None, ylim=None, side_by_side=False,
                      figsize_single=(10, 10), figsize_side=(9, 4),
                      dpi=600, save_path=None, cmap_name='nippy_spectral_r'):
    """
    Plot origin-centred trajectories coloured by max displacement.

    Parameters
    ----------
    data            : pd.DataFrame — combined dataframe with all conditions
    condition_col   : column name identifying the strain/condition
    particle_col    : column name for particle IDs
    x_col, y_col    : column names for x and y positions (in pixels)
    pixel_to_micron : conversion factor
    xlim, ylim      : tuple (min, max) in µm — set equal values for paper figures
                      e.g. xlim=(-50, 50). If None, auto-scales per plot.
    side_by_side    : if True, plots all conditions in one figure side by side
    figsize_single  : figure size for individual plots
    figsize_side    : figure size for side-by-side plot
    dpi             : resolution for saved figures
    save_path       : base path for saving, e.g. 'output/trajectories'
                      saves as '{save_path}_{condition}.png' or '{save_path}_side_by_side.png'
    cmap_name       : matplotlib colormap name
    """

    conditions = data[condition_col].unique()
    
    if cmap_name == 'dark_saturated':
        cmap = _dark_saturated_cmap()
    elif cmap_name == 'dark_saturated_r':
        cmap = _dark_saturated_cmap().reversed()
    else:
        cmap = cm.get_cmap(cmap_name)

    # ── Normalise each trajectory to origin, convert to µm ───────────────────
    processed = {}
    for cond in conditions:
        df = data[data[condition_col] == cond].copy()

        df['x_norm'] = df.groupby(particle_col)[x_col].transform(lambda x: x - x.iloc[0])
        df['y_norm'] = df.groupby(particle_col)[y_col].transform(lambda y: y - y.iloc[0])
        df['x_um']   = df['x_norm'] * pixel_to_micron
        df['y_um']   = df['y_norm'] * pixel_to_micron
        df['disp_um'] = np.sqrt(df['x_um']**2 + df['y_um']**2)

        max_disp = df.groupby(particle_col)['disp_um'].max()
        processed[cond] = (df, max_disp)

    # Use a shared colour scale across all conditions
    # all_max_disps = np.concatenate([md.values for _, md in processed.values()])
    # vmax_percentile = 95  # expose this as a function parameter
    # overall_max = np.percentile(all_max_disps, vmax_percentile)

    overall_max = max(md.max() for _, md in processed.values())
    norm = mcolors.Normalize(vmin=0, vmax=overall_max)

    def _draw_ax(ax, df, max_disp_um, title):
        ax.set_facecolor('white')
        for particle, group in df.groupby(particle_col):
            md    = max_disp_um[particle]
            color = cmap(norm(md))
            # alpha = 0.25 + 0.75 * (min(md, overall_max) / overall_max)
            alpha = 0.25 + 0.75 * (md / overall_max)
            ax.plot(group['x_um'], group['y_um'],
                    color=color, alpha=alpha, linewidth=0.7)

        sm = cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, fraction=0.03, pad=0.02)
        cbar.set_label('Max displacement (µm)', fontsize=11)

        ax.axhline(0, color='grey', linewidth=0.5, linestyle='--', alpha=0.5)
        ax.axvline(0, color='grey', linewidth=0.5, linestyle='--', alpha=0.5)
        ax.set_xlabel('Δx (µm)', fontsize=11)
        ax.set_ylabel('Δy (µm)', fontsize=11)
        ax.set_title(title, fontsize=13)
        ax.set_aspect('equal')

        if xlim: ax.set_xlim(xlim)
        if ylim: ax.set_ylim(ylim)

        n_particles = df[particle_col].nunique()
        ax.text(0.02, 0.98, f'n = {n_particles} cells',
                transform=ax.transAxes, fontsize=9,
                va='top', ha='left', color='grey')

    # ── Side-by-side ─────────────────────────────────────────────────────────
    if side_by_side:
        fig, axes = plt.subplots(1, len(conditions), figsize=figsize_side)
        if len(conditions) == 1:
            axes = [axes]
        for ax, cond in zip(axes, conditions):
            df, max_disp = processed[cond]
            _draw_ax(ax, df, max_disp, f'{cond} — trajectories centred at origin')
        plt.tight_layout()
        if save_path:
            fig.savefig(f'{save_path}_side_by_side.png', dpi=dpi, bbox_inches='tight')
            print(f"Saved: {save_path}_side_by_side.png")
        plt.show()

    # ── Individual plots ──────────────────────────────────────────────────────
    else:
        for cond in conditions:
            df, max_disp = processed[cond]
            fig, ax = plt.subplots(figsize=figsize_single)
            _draw_ax(ax, df, max_disp, f'{cond} — trajectories centred at origin')
            plt.tight_layout()
            if save_path:
                path = f'{save_path}_{cond}.png'
                fig.savefig(path, dpi=dpi, bbox_inches='tight')
                print(f"Saved: {path}")
            plt.show()

In [ ]:
def plot_maxdisp_histogram(data, condition_col='condition', particle_col='particle',
                           x_col='x', y_col='y', pixel_to_micron=0.166666,
                           conditions_to_plot=None,       # e.g. ['wt'] or ['wt', 'mutant']
                           min_displacement=None,          # e.g. 5.0 µm — filters out non-motile cells
                           vmax_percentile=95,
                           bins=50, xlim=None, ylim_hist=None,
                           colors=None, alpha_hist=0.4,
                           figsize=(10, 6), dpi=300, save_path=None):

    # ── Filter conditions ─────────────────────────────────────────────────────
    all_conditions = data[condition_col].unique()
    if conditions_to_plot is None:
        conditions = all_conditions
    else:
        conditions = [c for c in conditions_to_plot if c in all_conditions]
        missing = [c for c in conditions_to_plot if c not in all_conditions]
        if missing:
            print(f"WARNING: conditions not found in data: {missing}")

    if colors is None:
        default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
        colors = {cond: default_colors[i] for i, cond in enumerate(all_conditions)}

    # ── Compute max displacement per particle ─────────────────────────────────
    max_disps = {}
    for cond in conditions:
        df = data[data[condition_col] == cond].copy()

        df['x_norm']  = df.groupby(particle_col)[x_col].transform(lambda x: x - x.iloc[0])
        df['y_norm']  = df.groupby(particle_col)[y_col].transform(lambda y: y - y.iloc[0])
        df['x_um']    = df['x_norm'] * pixel_to_micron
        df['y_um']    = df['y_norm'] * pixel_to_micron
        df['disp_um'] = np.sqrt(df['x_um']**2 + df['y_um']**2)

        max_disp = df.groupby(particle_col)['disp_um'].max()

        # ── Minimum displacement filter ───────────────────────────────────────
        if min_displacement is not None:
            n_before = len(max_disp)
            max_disp = max_disp[max_disp >= min_displacement]
            n_after  = len(max_disp)
            n_removed = n_before - n_after
            pct = 100 * n_removed / n_before
            print(f"[{cond}] min_displacement={min_displacement} µm: "
                  f"kept {n_after}/{n_before} cells ({pct:.1f}% removed)")

        max_disps[cond] = max_disp.values

    # ── Axis limits ───────────────────────────────────────────────────────────
    all_vals = np.concatenate(list(max_disps.values()))
    x_min = (min_displacement if min_displacement is not None else 0) if xlim is None else xlim[0]
    x_max = np.percentile(all_vals, vmax_percentile)                   if xlim is None else xlim[1]
    kde_x = np.linspace(x_min, x_max * 1.05, 500)

    # ── Plot ──────────────────────────────────────────────────────────────────
    fig, ax = plt.subplots(figsize=figsize)

    for cond in conditions:
        vals  = max_disps[cond]
        color = colors[cond]
        n     = len(vals)

        ax.hist(vals, bins=bins, range=(x_min, x_max),
                density=False, alpha=alpha_hist,
                color=color, edgecolor=color, linewidth=0.5,
                label=f'{cond} (n={n})')

        # kde   = gaussian_kde(vals, bw_method='scott')
        # kde_y = kde(kde_x)
        # ax.plot(kde_x, kde_y, color=color, linewidth=2)

        med = np.median(vals)
        ax.axvline(med, color=color, linewidth=1.2, linestyle='--', alpha=0.8,
                   label=f'{cond} median: {med:.1f} µm')

    ax.set_xlabel('Max displacement (µm)', fontsize=12)
    ax.set_ylabel('Count', fontsize=12)

    title = 'Max displacement per cell'
    if min_displacement is not None:
        title += f'  [min disp ≥ {min_displacement} µm]'
    ax.set_title(title, fontsize=13)

    ax.legend(fontsize=10)
    if xlim:      ax.set_xlim(xlim)
    if ylim_hist: ax.set_ylim(ylim_hist)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=dpi, bbox_inches='tight')
        print(f"Saved: {save_path}")
    plt.show()

In [ ]:



# Both strains, filter out non-motile cells below 2 µm
plot_maxdisp_histogram(combined, pixel_to_micron=0.1666,
                       min_displacement=2,
                      save_path='figures/maxdisp_histogram.png')

